# Pronóstico del precio de bolsa de energía en Colombia

Notebook único y autocontenido: va del dato crudo al modelo evaluado, con los
métodos del curso de *Statistical Learning and Data Analysis* (SLDA).

**Dato de entrada:** un solo archivo, `panel_diario.csv` (4.018 días x 58
variables, 2015-2025), producido por `00_consolidar_panel.py` a partir de XM/SIMEM,
NOAA (índice ONI) y Banco de la República (TRM).

**Objetivo:** el cambio diario del log del precio,
$y_t = \log P_t - \log P_{t-1}$. El nivel se reconstruye como
$\hat P_t = P_{t-1}\,e^{\hat y_t}$.

## Contenido

| Sección | Tema | Referencia SLDA |
|---|---|---|
| 1 | Carga y análisis exploratorio | Clases 1-3 |
| 2 | Ingeniería de características | repo *feature-engineering* |
| 3 | Particiones temporales y error de predicción | Clase 4 (MSE, CV) |
| 4 | Filtros univariados: información mutua | repo *Complete-Feature-Selection* |
| 5 | Selección de variables: stepwise y best subset | Clases 5-7 |
| 6 | Penalización: Ridge, Lasso, Elastic Net | Clases 7-8 |
| 7 | Reducción de dimensión: PCR y PLS | Clases 6-7 |
| 8 | Votación y conjunto final de variables | — |
| 9 | Modelos aditivos: splines y GAM | Clases 10-11 |
| 10 | Árboles y boosting | Clase 15 |
| 11 | Evaluación final: walk-forward y Diebold-Mariano | Clase 4 |
| 12 | La dirección del precio como clasificación | Clases 12-14 |
| 13 | Conclusiones | — |

## Las tres ventanas temporales

Ninguna etapa ve datos de la etapa siguiente. Esta es la regla que hace creíbles
las métricas finales.

| Ventana | Fechas | Para qué se usa | Qué NO puede ver |
|---|---|---|---|
| **Selección** | 2015-01-01 a 2019-12-31 | elegir variables | 2020 en adelante |
| **Validación** | 2020-01-01 a 2022-12-31 | elegir modelo e hiperparámetros | 2023 en adelante |
| **Prueba** | 2023-01-01 a 2025-12-31 | **solo** medir el desempeño final | — |

Si la selección viera la validación, la comparación de modelos quedaría sesgada a
favor de lo seleccionado. Si la validación viera la prueba, las métricas finales
quedarían optimistas.

---
# 1. Carga y análisis exploratorio

In [ ]:
# En Colab ya vienen instaladas casi todas. Esta celda solo completa lo que falte.
import importlib.util, subprocess, sys

for paquete, modulo in [("xgboost", "xgboost"), ("statsmodels", "statsmodels")]:
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando {paquete}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)
print("Dependencias listas")

In [ ]:
import io
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf

from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import (LinearRegression, Lasso, LassoCV, RidgeCV,
                                  ElasticNetCV, lasso_path, LogisticRegressionCV)
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import (mean_squared_error, mean_absolute_error, roc_curve,
                             roc_auc_score, confusion_matrix, accuracy_score)
from sklearn.feature_selection import mutual_info_regression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.inspection import permutation_importance
import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

OBJETIVO = "precio_ponderado"

FIN_SELECCION  = "2019-12-31"
INI_VALIDACION = "2020-01-01"
FIN_VALIDACION = "2022-12-31"
INI_PRUEBA     = "2023-01-01"

SEMILLA = 42

# Validación cruzada que respeta el orden del tiempo: siempre entrena con el
# pasado y valida con el futuro. Un KFold normal entrenaría con datos de 2022
# para validar en 2016, lo que no tiene sentido en una serie temporal.
tscv = TimeSeriesSplit(n_splits=5)

### Carga del panel

La celda busca el archivo en tres lugares, en orden: disco local, repositorio de
GitHub, y si ninguno funciona, pide subirlo a mano. Así el mismo notebook corre
igual en tu computador y en Colab.

In [ ]:
ARCHIVO = "panel_diario_ampliado.csv"
URL = ("https://raw.githubusercontent.com/andrw4-4/"
       "pronostico-precio-bolsa-colombia/main/panel_diario_ampliado.csv")


def cargar_panel():
    for ruta in [Path(ARCHIVO), Path("data") / ARCHIVO, Path("/content") / ARCHIVO,
                 Path("/content/drive/MyDrive") / ARCHIVO]:
        if ruta.exists():
            print(f"Leído del disco: {ruta}")
            return pd.read_csv(ruta, parse_dates=["fecha"], index_col="fecha")

    try:
        datos = pd.read_csv(URL, parse_dates=["fecha"], index_col="fecha")
        print(f"Descargado de GitHub ({URL})")
        return datos
    except Exception as e:
        print(f"No se pudo descargar ({type(e).__name__}). Súbelo a mano.")

    from google.colab import files
    subidos = files.upload()
    nombre = next(iter(subidos))
    return pd.read_csv(io.BytesIO(subidos[nombre]), parse_dates=["fecha"], index_col="fecha")


df = cargar_panel().sort_index().asfreq("D")
print(f"{df.shape[0]} días x {df.shape[1]} variables  |  "
      f"{df.index.min().date()} a {df.index.max().date()}")
df.head()

### Datos faltantes

Las variables solares empiezan en 2021 y el precio de escasez inferior casi no se
publica: no se pueden usar en todo el periodo. Las de **eventos** (vertimientos,
demanda no atendida, importaciones, exportaciones) no tienen ningún cero en los
datos de XM, lo que indica que el día sin dato es un día **sin evento**, no un
dato perdido.

In [ ]:
faltantes = df.isna().mean().sort_values(ascending=False)
faltantes[faltantes > 0].round(3).to_frame("fracción faltante")

### La serie del precio

In [ ]:
precio = df[OBJETIVO]
log_precio = np.log(precio)
dlog = log_precio.diff()      # cambio diario del log ~ variación porcentual

fig, ax = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
precio.plot(ax=ax[0], color="k", lw=0.8, title="Precio de bolsa ponderado (COP/kWh)")
log_precio.plot(ax=ax[1], lw=0.8, title="Log del precio")
dlog.plot(ax=ax[2], lw=0.6, color="tab:red", title="Cambio diario del log (Δlog)")
for a in ax:
    a.axvline(pd.Timestamp(INI_VALIDACION), color="gray", ls="--")
    a.axvline(pd.Timestamp(INI_PRUEBA), color="gray", ls="--")
ax[0].text(pd.Timestamp("2016-06-01"), precio.max() * 0.9, "selección", color="gray")
ax[0].text(pd.Timestamp("2021-01-01"), precio.max() * 0.9, "validación", color="gray")
ax[0].text(pd.Timestamp("2023-10-01"), precio.max() * 0.9, "prueba", color="gray")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
precio.hist(bins=80, ax=ax[0]); ax[0].set_title("Precio: muy sesgado a la derecha")
log_precio.hist(bins=80, ax=ax[1]); ax[1].set_title("Log del precio: más simétrico")
dlog.hist(bins=120, ax=ax[2]); ax[2].set_title("Δlog: centrado en cero")
plt.tight_layout()
plt.show()

precio.describe().round(1).to_frame().T

### ¿Es estacionaria? (prueba ADF)

Los modelos lineales y la selección de variables suponen una serie estable en el
tiempo. La prueba ADF no rechaza la estacionariedad ni siquiera en el nivel, pero
el nivel tiene picos enormes (hasta 2.500 COP/kWh) y una varianza que cambia mucho
entre años.

Por eso se modela el **cambio diario del log** (Δlog):

- es claramente estacionario y su escala es comparable en todo el periodo;
- obliga al modelo a predecir el *movimiento* del precio en vez de copiar el de ayer.

Este segundo punto es el importante. Si se modela el nivel, un modelo que solo
repite `precio_lag1` ya obtiene un MAPE bajísimo sin haber aprendido nada.

In [ ]:
for nombre, serie in [("precio", precio), ("log precio", log_precio), ("Δlog precio", dlog)]:
    p = adfuller(serie.dropna(), autolag="AIC")[1]
    print(f"{nombre:12s} p-valor ADF = {p:.4f}  ->  "
          f"{'estacionaria' if p < 0.05 else 'NO estacionaria'}")

### Memoria y estacionalidad del cambio diario

La autocorrelación indica qué rezagos del propio precio vale la pena usar como
variables. El patrón semanal justifica las variables de calendario.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
plot_acf(dlog.dropna(), lags=30, ax=ax[0], title="Autocorrelación de Δlog")
dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
(dlog.groupby(dlog.index.dayofweek).mean().set_axis(dias)
     .plot.bar(ax=ax[1], title="Δlog promedio por día de la semana"))
plt.tight_layout()
plt.show()

### Relación de cada variable con el precio

Correlación de cada variable (medida ayer) con el log del precio de hoy,
**solo en la ventana de selección**. Mirar esto en todo el periodo ya sería usar
información de la prueba para decidir qué variables explorar.

In [ ]:
seleccion_cruda = df.loc[:FIN_SELECCION]
corr = (seleccion_cruda.shift(1)
        .corrwith(np.log(seleccion_cruda[OBJETIVO])).drop(OBJETIVO).dropna())
top = corr.reindex(corr.abs().sort_values(ascending=False).index).head(20)

top.sort_values().plot.barh(figsize=(8, 6),
                            color=np.where(top.sort_values() > 0, "tab:red", "tab:blue"))
plt.title("Top 20 correlaciones con el log del precio (ventana de selección)")
plt.show()

---
# 2. Ingeniería de características

**Regla anti-fuga:** para predecir el día $t$ solo se usa información disponible
antes de ese día. En la práctica:

| Tipo de variable | Rezago mínimo | Razón |
|---|---|---|
| Medidas durante el día (hidrología, demanda, generación) | 1 día | se conocen al cierre del día |
| *Ex ante* (precio de escasez, disponibilidad declarada, cargo por confiabilidad, margen de reserva, dummies CREG) | 0 días | se publican antes del despacho |
| Eventos (vertimientos, demanda no atendida, importaciones/exportaciones) | 1 día | además se rellenan con 0: sin dato = sin evento |
| Despacho (`costo_marginal_prog`, `precio_oferta_desp`, `max_precio_oferta`) | **excluidas** | ver abajo |

**Por qué se excluye el bloque de despacho.** El precio de bolsa *es* el costo
marginal del despacho ideal. Incluir `costo_marginal_prog` como variable da un
MAPE espectacular y vacío: el modelo simplemente copia una versión del precio, y
el aporte hidrológico -que es la pregunta de la tesis- queda invisible. Si se
quisiera un techo de precisión alcanzable con información de despacho, se
incluiría a propósito y se reportaría como *benchmark* superior, no como resultado.

**Una advertencia sobre `disp_comercial`.** XM la describe como la declaración
"modificada cuando se presentan cambios en las unidades de generación en operación
real", es decir, se ajusta *después* del despacho. Su valor del mismo día
correlaciona con el cambio de precio de ese día (-0,23) pero casi nada con el del
día siguiente (-0,02): la correlación es contable, no predictiva. Por eso se usa
la disponibilidad **declarada**, que sí va en la oferta del día anterior.

In [ ]:
DESPACHO = ["costo_marginal_prog", "precio_oferta_desp", "max_precio_oferta"]
EX_ANTE = ["precio_escasez", "precio_marg_escasez", "precio_escasez_empalmado",
           "disp_declarada", "enficc", "precio_cargo_conf", "cee", "cere", "mc",
           "margen_reserva", "dummy_creg_escasez",
           # Del panel ampliado: la disponibilidad se DECLARA antes del despacho,
           # igual que disp_declarada, así que puede entrar con rezago 0.
           "disp_decl_hidraulica", "disp_decl_termica", "disp_decl_despacho_central"]
EVENTOS = ["vertimientos", "demanda_no_atendida", "dem_no_atendida_noprog",
           "importaciones", "exportaciones",
           # Del panel ampliado: el ausente es un cero real, no un dato perdido.
           # En 2015 la generación solar del país era cero porque no había plantas;
           # en gen_liquidos el ausente es "ninguna térmica de líquidos despachó".
           "gen_solar", "frac_solar", "gen_eolica", "frac_eolica", "gen_liquidos"]
MAX_FALTANTE = 0.15

panel = df.copy()
# Se intersecta con las columnas presentes para que el notebook corra igual con el
# panel de 58 variables y con el ampliado de 104.
EX_ANTE = [c for c in EX_ANTE if c in panel.columns]
EVENTOS = [c for c in EVENTOS if c in panel.columns]
panel[EVENTOS] = panel[EVENTOS].fillna(0)      # sin dato = no hubo evento

# Una variable que empieza tarde le cuesta la muestra a TODAS las demás: el
# dropna() conjunto exige que todas tengan dato el mismo día, así que la de
# arranque más tardío fija el inicio del panel utilizable. vol_pct_el_quimbo
# (el embalse entró en operación en noviembre de 2015) costaba 10 meses de la
# ventana de selección ella sola.
INICIO_MAXIMO = pd.Timestamp("2015-03-01")
arranque = panel.apply(lambda s: s.first_valid_index())
tardias = [c for c in panel.columns
           if arranque[c] is not None and arranque[c] > INICIO_MAXIMO
           and panel[c].isna().mean() <= MAX_FALTANTE]

faltantes_panel = panel.isna().mean()
candidatas = [c for c in panel.columns
              if c != OBJETIVO and c not in DESPACHO and c not in tardias
              and faltantes_panel[c] <= MAX_FALTANTE]

if tardias:
    print(f"Se descartan por arrancar después de {INICIO_MAXIMO.date()} "
          f"(costarían muestra a todas las demás):")
    print(f"  {sorted(tardias)}")

print("Se descartan por faltantes o por ser del despacho:")
print(sorted(set(panel.columns) - set(candidatas) - {OBJETIVO}))

# Relleno hacia adelante: un día sin dato toma el último valor conocido.
# Solo mira hacia atrás, así que no introduce información del futuro.
base = panel[candidatas].ffill()

# Transformación no lineal explícita: con embalses bajos el precio sube mucho más
# que proporcionalmente. Es la única forma funcional que se impone a mano; en la
# sección 9 los splines estiman esa curvatura sin suponerla.
base["inv_volumen_util_pct"] = 1 / base["volumen_util_pct"].clip(lower=0.02)

ex_ante = [c for c in base.columns if c in EX_ANTE]
ex_post = [c for c in base.columns if c not in EX_ANTE]
print(f"\n{len(ex_ante)} variables ex ante y {len(ex_post)} medidas durante el día")

### Las tres familias de características

Para cada variable $x$ se crean tres tipos. Todas usan solo información hasta ayer:

| Tipo | Nombre | Qué mide |
|---|---|---|
| Rezago | `x_lag1` … `x_lag30` | el valor de hace 1, 2, 3, 7, 14 y 30 días |
| Media móvil | `x_ma7`, `x_ma30` | el nivel "normal" de la última semana y del último mes |
| Choque | `x_choque7`, `x_choque30` | cuánto se alejó el valor de ayer de su media móvil |

Las *ex ante* llevan además `x_lag0`. Para el precio se usan rezagos de su cambio
diario (`dlog_lagk`), su volatilidad (`dlog_sd7`, `dlog_sd30`) y los choques del
log del precio frente a su media móvil (`precio_choque7`, `precio_choque30`), que
capturan reversión a la media.

**Detalle técnico que importa:** toda media móvil se calcula con `shift(1)` *antes*
del `rolling`, nunca centrada. Una media móvil centrada en $t$ usa datos de
$t+1, t+2, \dots$ y es fuga de información pura. Es el error más común y el más
difícil de ver después.

In [ ]:
LAGS = [1, 2, 3, 7, 14, 30]
VENTANAS = [7, 30]


def crear_lags(datos, columnas, lags):
    nuevas = {f"{col}_lag{k}": datos[col].shift(k) for col in columnas for k in lags}
    return pd.DataFrame(nuevas, index=datos.index)


def crear_medias_moviles(datos, columnas, ventanas):
    nuevas = {f"{col}_ma{w}": datos[col].shift(1).rolling(w).mean()
              for col in columnas for w in ventanas}
    return pd.DataFrame(nuevas, index=datos.index)


def crear_choques(datos, columnas, ventanas):
    nuevas = {f"{col}_choque{w}": datos[col].shift(1) - datos[col].shift(1).rolling(w).mean()
              for col in columnas for w in ventanas}
    return pd.DataFrame(nuevas, index=datos.index)


def crear_features_precio(log_precio, lags, ventanas):
    dlog = log_precio.diff()
    nuevas = {f"dlog_lag{k}": dlog.shift(k) for k in lags}
    for w in ventanas:
        nuevas[f"dlog_sd{w}"] = dlog.shift(1).rolling(w).std()
        nuevas[f"precio_choque{w}"] = (log_precio.shift(1)
                                       - log_precio.shift(1).rolling(w).mean())
    return pd.DataFrame(nuevas, index=log_precio.index)


def crear_calendario(indice):
    return pd.DataFrame({
        "es_lunes": (indice.dayofweek == 0).astype(int),
        "es_sabado": (indice.dayofweek == 5).astype(int),
        "es_domingo": (indice.dayofweek == 6).astype(int),
    }, index=indice)

In [ ]:
X = pd.concat([
    crear_features_precio(log_precio, LAGS, VENTANAS),
    crear_lags(base, ex_ante, [0] + LAGS),
    crear_lags(base, ex_post, LAGS),
    crear_choques(base, base.columns, VENTANAS),
    crear_medias_moviles(base, base.columns, VENTANAS),
    crear_calendario(df.index),
], axis=1)

y = dlog.rename("y")
datos = pd.concat([X, y], axis=1).dropna()   # se pierden ~60 días iniciales
print(f"Características creadas: {X.shape[1]}")
print(f"Observaciones utilizables: {len(datos)} "
      f"({datos.index.min().date()} a {datos.index.max().date()})")

### Limpieza previa a la selección

Tres filtros, en orden. Todos se calculan **solo con la ventana de selección**:

1. **Constantes.** Una variable sin variación no puede explicar nada.
2. **Redundantes.** Si dos variables tienen correlación mayor a 0,95 se conserva la
   primera. `volumen_util_pct_lag1`, `volumen_util_pct_lag2` y
   `volumen_util_pct_ma7` son prácticamente la misma variable.
3. **Combinaciones lineales exactas.** Por construcción
   `x_choque7 = x_lag1 − x_ma7`: con las tres juntas la matriz es singular y la
   regresión no puede separar sus efectos. Una variable se conserva solo si
   aumenta el rango de la matriz. Como los choques van antes que las medias
   móviles en el orden de las columnas, se conserva el choque, que es la
   variable interpretable ("se salió de lo normal").

Este paso es lo que en el curso se llama el problema de la **multicolinealidad**:
no sesga las predicciones, pero infla las varianzas de los coeficientes y vuelve
inestable cualquier método que los interprete.

In [ ]:
def quitar_redundantes(X, umbral=0.95):
    corr = X.corr().abs()
    quitar = set()
    for i, a in enumerate(X.columns):
        if a in quitar:
            continue
        for b in X.columns[i + 1:]:
            if corr.loc[a, b] > umbral:
                quitar.add(b)
    return [c for c in X.columns if c not in quitar]


def quitar_combinaciones_exactas(X):
    Z = ((X - X.mean()) / X.std()).values
    elegidas = []
    for i, _ in enumerate(X.columns):
        if np.linalg.matrix_rank(Z[:, elegidas + [i]]) > len(elegidas):
            elegidas.append(i)
    return [X.columns[i] for i in elegidas]

---
# 3. Particiones temporales y medida del error

Aquí se materializan las tres ventanas. A partir de este punto, cada método
declara explícitamente qué ventana puede ver.

In [ ]:
sel    = datos.loc[:FIN_SELECCION]
val    = datos.loc[INI_VALIDACION:FIN_VALIDACION]
prueba = datos.loc[INI_PRUEBA:]

X_sel_todas = sel.drop(columns="y")
no_constantes  = X_sel_todas.columns[X_sel_todas.std() > 0]
no_redundantes = quitar_redundantes(X_sel_todas[no_constantes])
COLUMNAS       = quitar_combinaciones_exactas(X_sel_todas[no_redundantes])

print(f"{X_sel_todas.shape[1]} creadas -> {len(no_constantes)} sin constantes -> "
      f"{len(no_redundantes)} sin redundantes -> {len(COLUMNAS)} sin combinaciones exactas\n")
for nombre, parte in [("Selección", sel), ("Validación", val), ("Prueba", prueba)]:
    print(f"{nombre:11s} {len(parte):5d} días   "
          f"{parte.index.min().date()} a {parte.index.max().date()}")

### Estandarización

Los métodos de penalización castigan el tamaño de los coeficientes, así que el
resultado depende de las unidades: una variable en GWh y otra en porcentaje no son
comparables. Se estandariza (media 0, desviación 1) **con los parámetros de la
ventana de selección**, y esos mismos parámetros se aplican a validación y prueba.

Calcular la media y la desviación sobre todo el periodo sería una fuga sutil pero
real: el modelo estaría usando el promedio del futuro para centrar el pasado.

In [ ]:
escalador = StandardScaler().fit(sel[COLUMNAS])

Xs_sel    = pd.DataFrame(escalador.transform(sel[COLUMNAS]),    index=sel.index,    columns=COLUMNAS)
Xs_val    = pd.DataFrame(escalador.transform(val[COLUMNAS]),    index=val.index,    columns=COLUMNAS)
Xs_prueba = pd.DataFrame(escalador.transform(prueba[COLUMNAS]), index=prueba.index, columns=COLUMNAS)

y_sel, y_val, y_prueba = sel["y"], val["y"], prueba["y"]

### Las métricas

Cuatro medidas, porque ninguna sola cuenta la historia completa:

- **RMSE Δlog:** error cuadrático en el cambio diario. Es lo que los modelos
  minimizan. Castiga mucho los errores grandes.
- **MAE Δlog:** error absoluto. Menos sensible a los picos.
- **Acierto de dirección (%):** porcentaje de días en que se acierta si el precio
  sube o baja. Para un operador esto puede importar más que el error en magnitud.
- **MAPE nivel (%):** error porcentual en COP/kWh, tras reconstruir el precio.
  Es la métrica interpretable para el sector.

**El rival siempre es el modelo ingenuo:** "el precio de mañana es igual al de
hoy" ($\hat y = 0$). Un MAPE de 8% suena bien hasta que se descubre que el
ingenuo logra 8,1%. Sin este punto de comparación, ninguna métrica significa nada.

In [ ]:
def construir_evaluador(indice, y_real):
    precio_ayer = precio.shift(1).loc[indice]
    precio_real = precio.loc[indice]

    def evaluar(pred):
        pred = pd.Series(np.ravel(pred), index=indice)
        nivel = precio_ayer * np.exp(pred)
        acierto = (np.nan if (pred == 0).all()
                   else 100 * np.mean(np.sign(pred) == np.sign(y_real)))
        return {"RMSE Δlog": np.sqrt(mean_squared_error(y_real, pred)),
                "MAE Δlog": mean_absolute_error(y_real, pred),
                "Acierto dirección (%)": acierto,
                "MAPE nivel (%)": 100 * np.mean(np.abs(nivel - precio_real) / precio_real)}
    return evaluar


evaluar_val    = construir_evaluador(val.index, y_val)
evaluar_prueba = construir_evaluador(prueba.index, y_prueba)

---
# 4. Filtros univariados: información mutua

Del repositorio *Complete-Feature-Selection*. Un filtro mira cada variable por
separado, sin modelo: es barato y sirve de tamizaje antes de los métodos caros.

La **información mutua** mide cuánta información aporta $X$ sobre $Y$, sin suponer
que la relación sea lineal:

$$I(X;Y) = \int\int p(x,y)\,\log\frac{p(x,y)}{p(x)p(y)}\,dx\,dy$$

Vale 0 si son independientes y crece con cualquier tipo de dependencia. **Esto es
crítico aquí:** la relación reservas-precio es convexa (con embalses bajos el
precio se dispara), así que la correlación de Pearson *subestima* la utilidad del
bloque hidrológico. La información mutua no.

**Limitación:** es univariada. No ve que dos variables juntas expliquen algo que
ninguna explica sola, y premia a las variables que sirven de reloj (una tendencia
suave tiene alta MI con cualquier serie con tendencia). Por eso es tamizaje, no
veredicto.

In [ ]:
mi = pd.Series(
    mutual_info_regression(Xs_sel, y_sel, random_state=SEMILLA),
    index=COLUMNAS, name="info_mutua"
).sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
mi.head(20).sort_values().plot.barh(ax=ax[0], color="tab:purple",
                                    title="Top 20 por información mutua")
correl = Xs_sel.corrwith(y_sel).abs()
ax[1].scatter(correl, mi, s=8, alpha=0.5)
ax[1].set_xlabel("|correlación| con Δlog"); ax[1].set_ylabel("información mutua")
ax[1].set_title("MI vs correlación: los puntos arriba-izquierda\nson relaciones no lineales")
plt.tight_layout()
plt.show()

no_lineales = (mi / mi.max() - correl / correl.max()).sort_values(ascending=False)
print("Variables con más señal no lineal que lineal:")
print(no_lineales.head(8).round(3).to_string())

---
# 5. Selección de variables: stepwise y best subset

*(SLDA clases 5-7. En R esto es `leaps::regsubsets`.)*

El problema: con $p$ variables hay $2^p$ modelos posibles. Con las más de doscientas
que quedaron tras la limpieza, eso es más que átomos en el universo observable. Los
métodos por pasos recorren una fracción de ese espacio de forma golosa.

**El criterio es el BIC**, no el $R^2$. El $R^2$ nunca baja al agregar variables,
así que siempre elegiría el modelo completo. El BIC castiga la complejidad:

$$\text{BIC} = -2\log L + k\log n$$

Frente al AIC (que usa $2k$), el BIC castiga más fuerte cuando $n$ es grande
-aquí $n \approx 1.800$, así que $\log n \approx 7,5$ contra 2- y tiende a elegir
modelos más pequeños. Para una tesis que busca **interpretar** qué mueve el
precio, un modelo pequeño y defendible vale más que uno grande y marginalmente
mejor en ajuste.

- **Forward:** parte del modelo vacío y agrega la variable que más baja el BIC.
- **Backward:** parte del modelo completo y quita en cada paso la menos
  significativa (menor $|t|$), mientras el BIC siga bajando.
- **Híbrido:** como forward, pero tras cada entrada revisa si alguna ya incluida sobra.

In [ ]:
def ajustar_ols(X, y, columnas):
    Z = X[columnas].copy()
    Z.insert(0, "const", 1.0)
    return sm.OLS(y, Z).fit()


def forward_stepwise(X, y, criterio="bic"):
    elegidas, restantes = [], list(X.columns)
    mejor = getattr(ajustar_ols(X, y, elegidas), criterio)
    while restantes:
        puntajes = {c: getattr(ajustar_ols(X, y, elegidas + [c]), criterio) for c in restantes}
        candidata = min(puntajes, key=puntajes.get)
        if puntajes[candidata] >= mejor:
            break
        mejor = puntajes[candidata]
        elegidas.append(candidata)
        restantes.remove(candidata)
    return elegidas


def backward_stepwise(X, y, criterio="bic"):
    elegidas = list(X.columns)
    modelo = ajustar_ols(X, y, elegidas)
    while len(elegidas) > 1:
        peor = modelo.tvalues.drop("const").abs().idxmin()
        sin_peor = ajustar_ols(X, y, [c for c in elegidas if c != peor])
        if getattr(sin_peor, criterio) > getattr(modelo, criterio):
            break
        elegidas.remove(peor)
        modelo = sin_peor
    return elegidas


def stepwise_hibrido(X, y, criterio="bic"):
    elegidas = []
    mejor = getattr(ajustar_ols(X, y, elegidas), criterio)
    while True:
        restantes = [c for c in X.columns if c not in elegidas]
        if not restantes:
            break
        puntajes = {c: getattr(ajustar_ols(X, y, elegidas + [c]), criterio) for c in restantes}
        candidata = min(puntajes, key=puntajes.get)
        if puntajes[candidata] >= mejor:
            break
        mejor = puntajes[candidata]
        elegidas.append(candidata)
        for c in elegidas[:-1]:                      # paso hacia atrás
            sin_c = [v for v in elegidas if v != c]
            valor = getattr(ajustar_ols(X, y, sin_c), criterio)
            if valor < mejor:
                mejor, elegidas = valor, sin_c
    return elegidas

In [ ]:
sel_forward  = forward_stepwise(Xs_sel, y_sel)
sel_backward = backward_stepwise(Xs_sel, y_sel)
sel_hibrido  = stepwise_hibrido(Xs_sel, y_sel)

for nombre, s in [("Forward", sel_forward), ("Backward", sel_backward), ("Híbrido", sel_hibrido)]:
    r2 = ajustar_ols(Xs_sel, y_sel, s).rsquared
    print(f"{nombre:9s} {len(s):2d} variables  R² = {r2:.3f}\n          {s}\n")

Resumen del modelo forward. Los coeficientes están estandarizados, así que su
tamaño sí es comparable entre variables: cada uno dice cuánto cambia Δlog ante un
cambio de una desviación estándar en esa variable.

In [ ]:
print(ajustar_ols(Xs_sel, y_sel, sel_forward).summary().tables[1])

## 5.2 Selección exhaustiva (best subset)

Evaluar **todas** las combinaciones de las variables disponibles es imposible.
La búsqueda exhaustiva se hace sobre las **12 variables** que ya propuso el
stepwise: $2^{12}-1 = 4.095$ modelos, que sí se pueden recorrer.

- Para cada tamaño $k$ se guarda el mejor modelo según $R^2$ (dentro de un mismo
  $k$, el $R^2$ sí es comparable porque la complejidad es igual).
- Entre esos mejores, se elige el $k$ con menor BIC.

Esta es exactamente la lógica de `regsubsets` + `which.min(BIC)` del curso.

In [ ]:
pool = list(dict.fromkeys(sel_forward + sel_hibrido + sel_backward))[:12]
print("Variables en la búsqueda:", pool)

subconjuntos = []
for k in range(1, len(pool) + 1):
    for combo in combinations(pool, k):
        m = ajustar_ols(Xs_sel, y_sel, list(combo))
        subconjuntos.append({"k": k, "variables": list(combo),
                             "r2": m.rsquared, "aic": m.aic, "bic": m.bic})
subconjuntos = pd.DataFrame(subconjuntos)

mejor_por_k = subconjuntos.loc[subconjuntos.groupby("k")["r2"].idxmax()].set_index("k")
sel_best = mejor_por_k.loc[mejor_por_k["bic"].idxmin(), "variables"]

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for a, col in zip(ax, ["r2", "aic", "bic"]):
    mejor_por_k[col].plot(ax=a, marker="o", title=f"{col.upper()} del mejor modelo de tamaño k")
    a.set_xlabel("k (número de variables)")
ax[2].axvline(mejor_por_k["bic"].idxmin(), color="red", ls="--")
plt.tight_layout()
plt.show()

print(f"Best subset elige {len(sel_best)} variables: {sel_best}")

---
# 6. Penalización: Ridge, Lasso y Elastic Net

*(SLDA clases 7-8. En R, `glmnet` con `alpha` entre 0 y 1.)*

En vez de buscar entre subconjuntos, se ajusta **un solo modelo** con un castigo
al tamaño de los coeficientes:

$$\hat\beta = \arg\min_\beta \;\|y - X\beta\|^2 \;+\; \lambda\Big[(1-\alpha)\tfrac{1}{2}\|\beta\|_2^2 + \alpha\|\beta\|_1\Big]$$

- **Ridge** ($\alpha=0$): encoge los coeficientes pero **ninguno llega a cero**.
  No selecciona; sirve contra la multicolinealidad, porque reparte el peso entre
  variables correlacionadas en vez de darle todo a una al azar.
- **Lasso** ($\alpha=1$): la geometría de la norma $\ell_1$ tiene esquinas, y el
  óptimo suele caer en una de ellas. Por eso lleva coeficientes **exactamente a
  cero**: selecciona.
- **Elastic Net** ($0<\alpha<1$): mezcla ambas. Con grupos de variables
  correlacionadas -exactamente nuestro caso- el Lasso elige una al azar y
  descarta el resto; el Elastic Net tiende a conservar el grupo.

$\lambda$ se elige con validación cruzada **temporal**, no aleatoria.

*Nota:* el **grouped lasso** de la clase 9 no está en scikit-learn. Sería el
método natural aquí, con los bloques temáticos (hidrología, oferta, demanda,
regulatorio) como grupos, para que el modelo incluya o excluya un bloque completo.
Queda anotado como extensión.

In [ ]:
lasso = LassoCV(cv=tscv, max_iter=50000, random_state=SEMILLA).fit(Xs_sel, y_sel)
coef_lasso = pd.Series(lasso.coef_, index=COLUMNAS)
sel_lasso = coef_lasso[coef_lasso != 0].abs().sort_values(ascending=False).index.tolist()
print(f"Lasso: alpha = {lasso.alpha_:.5f}, {len(sel_lasso)} de {len(COLUMNAS)} "
      f"variables con coeficiente distinto de cero")

alphas, coefs, _ = lasso_path(Xs_sel, y_sel, n_alphas=100)
plt.figure(figsize=(11, 4))
plt.plot(alphas, coefs.T, lw=0.8)
plt.axvline(lasso.alpha_, color="k", ls="--", label="alpha elegido por CV")
plt.xscale("log"); plt.xlabel("alpha"); plt.ylabel("coeficiente"); plt.legend()
plt.title("Trayectoria de Lasso: al crecer alpha, los coeficientes llegan a cero")
plt.show()

In [ ]:
ridge = RidgeCV(alphas=np.logspace(-2, 5, 80), cv=tscv).fit(Xs_sel, y_sel)
coef_ridge = pd.Series(ridge.coef_, index=COLUMNAS)
coef_ols = pd.Series(LinearRegression().fit(Xs_sel, y_sel).coef_, index=COLUMNAS)
print(f"Ridge: alpha = {ridge.alpha_:.2f}, coeficientes exactamente en 0: "
      f"{(coef_ridge == 0).sum()}")

top = coef_ridge.abs().sort_values(ascending=False).head(15).index
pd.DataFrame({"OLS (sin castigo)": coef_ols[top], "Ridge": coef_ridge[top]}).plot.barh(figsize=(9, 5))
plt.title("Ridge encoge los coeficientes, pero no los anula")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
enet = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], cv=tscv,
                    max_iter=50000, random_state=SEMILLA).fit(Xs_sel, y_sel)
coef_enet = pd.Series(enet.coef_, index=COLUMNAS)
sel_enet = coef_enet[coef_enet != 0].abs().sort_values(ascending=False).index.tolist()
print(f"Elastic Net: alpha = {enet.alpha_:.5f}, l1_ratio = {enet.l1_ratio_}, "
      f"{len(sel_enet)} variables seleccionadas")

---
# 7. Reducción de dimensión: PCR y PLS

*(SLDA clases 6-7. En R, `pls::pcr` y `pls::plsr`.)*

Otra estrategia: en vez de elegir variables originales, construir **componentes**
-combinaciones lineales de todas- y regresar sobre unos pocos.

- **PCR:** los componentes son los de PCA, que maximizan la varianza de $X$ **sin
  mirar $y$**. Es no supervisado, y ahí está su debilidad: la dirección de máxima
  varianza no tiene por qué ser la que predice.
- **PLS:** los componentes maximizan la covarianza con $y$. Es supervisado, así
  que suele necesitar menos componentes para el mismo error.

Ambos resuelven la multicolinealidad por construcción (los componentes son
ortogonales), pero **pierden interpretabilidad**: un componente es una mezcla de
todas las variables. Para esta tesis eso es un costo alto, porque la pregunta es *qué*
mueve el precio, no solo *cuánto* se puede predecir. Por eso entran como
comparación, no como modelo final.

In [ ]:
def rmse_cv(modelo, X, y):
    return -cross_val_score(modelo, X, y, cv=tscv,
                            scoring="neg_root_mean_squared_error").mean()


n_componentes = range(1, 31)
error_pcr = [rmse_cv(make_pipeline(PCA(n), LinearRegression()), Xs_sel, y_sel) for n in n_componentes]
error_pls = [rmse_cv(PLSRegression(n, scale=False), Xs_sel, y_sel) for n in n_componentes]

n_pcr = n_componentes[int(np.argmin(error_pcr))]
n_pls = n_componentes[int(np.argmin(error_pls))]

plt.figure(figsize=(10, 4))
plt.plot(n_componentes, error_pcr, marker="o", ms=3, label=f"PCR (mejor: {n_pcr})")
plt.plot(n_componentes, error_pls, marker="o", ms=3, label=f"PLS (mejor: {n_pls})")
plt.xlabel("número de componentes"); plt.ylabel("RMSE en validación cruzada"); plt.legend()
plt.title("¿Cuántos componentes usar?")
plt.show()

pca = PCA().fit(Xs_sel)
print(f"PCR: {n_pcr} componentes explican el "
      f"{100 * pca.explained_variance_ratio_[:n_pcr].sum():.1f}% de la varianza de X")

PCR y PLS no eliminan variables, pero sus *loadings* muestran qué variables forman
cada componente. Estas son las de mayor peso en el primer componente de PLS:

In [ ]:
pls = PLSRegression(n_pls, scale=False).fit(Xs_sel, y_sel)
pesos_pls = pd.Series(pls.x_weights_[:, 0], index=COLUMNAS)
(pesos_pls.reindex(pesos_pls.abs().sort_values(ascending=False).index)
          .head(12).round(3).to_frame("peso en el componente 1"))

---
# 8. Votación y conjunto final de variables

Ningún método decide solo. Cada uno tiene sus sesgos: el stepwise es goloso y
depende del orden; el Lasso elige arbitrariamente dentro de grupos correlacionados;
el best subset solo vio 12 candidatas.

Cada método que **selecciona** variables aporta un voto (Ridge, PCR y PLS no votan
porque no seleccionan). Pasan al modelo las variables con **al menos 3 votos de 6**.

La lógica: una variable que aparece bajo criterios distintos -información,
significancia, penalización- es mucho más difícil de explicar como azar que una
que solo un método eligió.

In [ ]:
metodos = {"Forward": sel_forward, "Backward": sel_backward, "Híbrido": sel_hibrido,
           "Best subset": sel_best, "Lasso": sel_lasso, "Elastic Net": sel_enet}

votos = pd.DataFrame({m: [int(c in s) for c in COLUMNAS] for m, s in metodos.items()},
                     index=COLUMNAS)
votos["total"] = votos.sum(axis=1)
votos["info_mutua"] = mi.reindex(COLUMNAS).round(3)
votos = votos[votos["total"] > 0].sort_values("total", ascending=False)

MIN_VOTOS = 3
FEATURES_FINALES = votos[votos["total"] >= MIN_VOTOS].index.tolist()
print(f"Variables finales ({len(FEATURES_FINALES)}):")
for v in FEATURES_FINALES:
    print(f"  {votos.loc[v, 'total']}/6 votos   {v}")
votos.head(30)

### Comparación de los modelos lineales en **validación** (2020-2022)

Todos se entrenan en la ventana de selección y se miden en la de validación. La
prueba sigue intacta. Esta tabla es la que decide qué familia de modelos vale la
pena llevar a la evaluación final.

In [ ]:
def predecir_ols(columnas, X_entrena, y_entrena, X_predice):
    return LinearRegression().fit(X_entrena[columnas], y_entrena).predict(X_predice[columnas])


tabla_val = pd.DataFrame({
    "Ingenuo (precio de ayer)": evaluar_val(np.zeros(len(val))),
    "OLS forward":      evaluar_val(predecir_ols(sel_forward,  Xs_sel, y_sel, Xs_val)),
    "OLS backward":     evaluar_val(predecir_ols(sel_backward, Xs_sel, y_sel, Xs_val)),
    "OLS híbrido":      evaluar_val(predecir_ols(sel_hibrido,  Xs_sel, y_sel, Xs_val)),
    "OLS best subset":  evaluar_val(predecir_ols(sel_best,     Xs_sel, y_sel, Xs_val)),
    "OLS votadas":      evaluar_val(predecir_ols(FEATURES_FINALES, Xs_sel, y_sel, Xs_val)),
    "Lasso":            evaluar_val(lasso.predict(Xs_val)),
    "Ridge":            evaluar_val(ridge.predict(Xs_val)),
    "Elastic Net":      evaluar_val(enet.predict(Xs_val)),
    f"PCR ({n_pcr} comp.)": evaluar_val(
        make_pipeline(PCA(n_pcr), LinearRegression()).fit(Xs_sel, y_sel).predict(Xs_val)),
    f"PLS ({n_pls} comp.)": evaluar_val(pls.predict(Xs_val)),
}).T
tabla_val.round(4).sort_values("RMSE Δlog")

---
# 9. Modelos aditivos: splines y GAM

*(SLDA clases 10-11.)*

Hasta aquí todo fue lineal. Pero ya sabemos que la relación reservas-precio **no**
lo es: con embalses bajos el precio se dispara. De hecho, en la sección 2 se
impuso esa forma a mano con `1/volumen_util_pct`.

Un **modelo aditivo generalizado (GAM)** estima esa curvatura sin suponerla:

$$y = \beta_0 + f_1(x_1) + f_2(x_2) + \dots + f_p(x_p) + \varepsilon$$

Cada $f_j$ es una función suave estimada de los datos. Se mantiene la
**aditividad** -no hay interacciones-, así que el modelo sigue siendo
interpretable: se puede graficar $f_j$ y leer qué le hace esa variable al precio.

**Implementación:** cada variable se expande en una base de *B-splines* y se ajusta
un Ridge sobre la base expandida. Como no se crean productos entre variables, el
resultado es aditivo por construcción: esto **es** un GAM. La penalización Ridge
cumple el papel del parámetro de suavizado.

El número de nodos controla la flexibilidad y es el clásico compromiso
sesgo-varianza: pocos nodos, el modelo no captura la curvatura (sesgo); muchos,
ajusta ruido (varianza).

In [ ]:
def construir_gam(n_nodos):
    return make_pipeline(
        StandardScaler(),
        SplineTransformer(n_knots=n_nodos, degree=3, include_bias=False),
        RidgeCV(alphas=np.logspace(-3, 4, 60)),
    )


nodos_probados = [3, 4, 5, 6, 8, 10]
error_gam = [rmse_cv(construir_gam(k), sel[FEATURES_FINALES], y_sel) for k in nodos_probados]
n_nodos = nodos_probados[int(np.argmin(error_gam))]

# Lineal sobre las mismas variables, como referencia
error_lineal = rmse_cv(make_pipeline(StandardScaler(), LinearRegression()),
                       sel[FEATURES_FINALES], y_sel)

plt.figure(figsize=(9, 3.5))
plt.plot(nodos_probados, error_gam, marker="o", label="GAM (splines)")
plt.axhline(error_lineal, color="gray", ls="--", label="lineal")
plt.xlabel("número de nodos por variable"); plt.ylabel("RMSE en validación cruzada")
plt.title("Flexibilidad del GAM: el compromiso sesgo-varianza"); plt.legend()
plt.show()

gam = construir_gam(n_nodos).fit(sel[FEATURES_FINALES], y_sel)
print(f"GAM elegido: {n_nodos} nodos por variable, "
      f"{gam[1].transform(sel[FEATURES_FINALES]).shape[1]} funciones base en total")

### ¿Qué forma tiene cada efecto?

Se grafica el efecto parcial de las variables más importantes: se varía una sola
variable sobre su rango observado, con las demás fijas en su mediana. Si la curva
sale recta, el efecto era lineal y el spline no aportaba nada; si sale curva, ahí
había no linealidad que los modelos lineales estaban aproximando mal.

In [ ]:
importantes = FEATURES_FINALES[:6]
medianas = sel[FEATURES_FINALES].median()

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for a, var in zip(axes.ravel(), importantes):
    rejilla = np.linspace(sel[var].quantile(0.01), sel[var].quantile(0.99), 200)
    Xg = pd.DataFrame([medianas] * len(rejilla), columns=FEATURES_FINALES)
    Xg[var] = rejilla
    a.plot(rejilla, gam.predict(Xg), color="tab:green")
    a.set_title(var, fontsize=9); a.set_xlabel(""); a.set_ylabel("efecto en Δlog")
for a in axes.ravel()[len(importantes):]:
    a.axis("off")
plt.suptitle("Efectos parciales estimados por el GAM", y=1.01)
plt.tight_layout()
plt.show()

---
# 10. Árboles y boosting

*(SLDA clase 15. En R, `rpart` + `prune` por `cp`.)*

Un árbol parte el espacio en regiones con cortes del tipo "¿`aportes_lag1` < 0,6?"
y predice la media de cada región. Ventajas: captura no linealidades e
interacciones sin que se las especifiquen, y no necesita estandarización.
Desventaja: **alta varianza**: cambiar unos pocos datos cambia el árbol entero.

**Poda por costo-complejidad.** Un árbol sin restricción memoriza el entrenamiento.
Se penaliza el número de hojas:

$$C_\alpha(T) = \sum_{\text{hojas}} \text{SSE} \;+\; \alpha\,|T|$$

Es exactamente el parámetro `cp` de `rpart`; en scikit-learn se llama `ccp_alpha`
y se elige por validación cruzada.

Vale la pena mirar cuántas hojas deja la poda. Con un objetivo tan ruidoso como
Δlog, la validación cruzada suele quedarse con un árbol grande: las diferencias de
error entre árboles son pequeñas frente al ruido, así que el criterio no logra
distinguirlos bien. Ese es justamente el síntoma de la **alta varianza** del árbol
único, y el argumento para pasar a boosting.

**Boosting** es la respuesta a esa varianza: en vez de un árbol profundo, se suman
cientos de árboles **poco profundos**, cada uno ajustando el error que dejó la suma
de los anteriores. XGBoost es esa idea con regularización y una implementación
eficiente.

In [ ]:
# Ruta de poda: cada alpha da un árbol progresivamente más pequeño
arbol_completo = DecisionTreeRegressor(random_state=SEMILLA)
ruta = arbol_completo.cost_complexity_pruning_path(sel[FEATURES_FINALES], y_sel)
alphas_poda = np.unique(ruta.ccp_alphas[ruta.ccp_alphas > 0])
alphas_poda = alphas_poda[::max(1, len(alphas_poda) // 25)][:25]

error_arbol = [rmse_cv(DecisionTreeRegressor(ccp_alpha=a, random_state=SEMILLA),
                       sel[FEATURES_FINALES], y_sel) for a in alphas_poda]
mejor_alpha = alphas_poda[int(np.argmin(error_arbol))]

plt.figure(figsize=(9, 3.5))
plt.plot(alphas_poda, error_arbol, marker="o", ms=3)
plt.axvline(mejor_alpha, color="red", ls="--", label=f"alpha = {mejor_alpha:.2e}")
plt.xscale("log"); plt.xlabel("ccp_alpha (equivalente al cp de rpart)")
plt.ylabel("RMSE en validación cruzada"); plt.legend()
plt.title("Poda por costo-complejidad")
plt.show()

arbol = DecisionTreeRegressor(ccp_alpha=mejor_alpha, random_state=SEMILLA).fit(
    sel[FEATURES_FINALES], y_sel)
print(f"Árbol podado: {arbol.get_n_leaves()} hojas, profundidad {arbol.get_depth()}")

plt.figure(figsize=(16, 6))
plot_tree(arbol, feature_names=FEATURES_FINALES, max_depth=2, filled=True,
          fontsize=8, precision=3)
plt.title("Primeros niveles del árbol podado")
plt.show()

### XGBoost

Árboles poco profundos (`max_depth=3`) y tasa de aprendizaje baja (0,02): muchas
correcciones pequeñas en vez de pocas grandes. El número de árboles se decide con
*early stopping* **en la ventana de validación**, que es justamente para lo que
existe esa ventana.

In [ ]:
PARAMS = dict(learning_rate=0.02, max_depth=3, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=5, random_state=SEMILLA, n_jobs=-1, tree_method="hist")


def entrenar_xgb(columnas, entrena, valida, etiqueta=""):
    buscador = xgb.XGBRegressor(n_estimators=3000, early_stopping_rounds=100, **PARAMS)
    buscador.fit(entrena[columnas], entrena["y"],
                 eval_set=[(valida[columnas], valida["y"])], verbose=False)
    n_arboles = buscador.best_iteration + 1
    print(f"{etiqueta:24s} {len(columnas):3d} variables -> {n_arboles} árboles")
    return n_arboles


N_ARBOLES_SEL    = entrenar_xgb(FEATURES_FINALES, sel, val, "XGBoost (seleccionadas)")
N_ARBOLES_TODAS  = entrenar_xgb(COLUMNAS,         sel, val, "XGBoost (todas)")

# Entrenados en selección, medidos en validación
xgb_sel_val = xgb.XGBRegressor(n_estimators=N_ARBOLES_SEL, **PARAMS).fit(
    sel[FEATURES_FINALES], y_sel)
xgb_todas_val = xgb.XGBRegressor(n_estimators=N_ARBOLES_TODAS, **PARAMS).fit(
    sel[COLUMNAS], y_sel)

tabla_val.loc["Árbol podado"] = pd.Series(evaluar_val(arbol.predict(val[FEATURES_FINALES])))
tabla_val.loc["GAM (splines)"] = pd.Series(evaluar_val(gam.predict(val[FEATURES_FINALES])))
tabla_val.loc["XGBoost (seleccionadas)"] = pd.Series(evaluar_val(xgb_sel_val.predict(val[FEATURES_FINALES])))
tabla_val.loc["XGBoost (todas)"] = pd.Series(evaluar_val(xgb_todas_val.predict(val[COLUMNAS])))

print("\nComparación completa en VALIDACIÓN (2020-2022):")
tabla_val.round(4).sort_values("RMSE Δlog")

---
# 11. Evaluación final: walk-forward y Diebold-Mariano

Hasta aquí la prueba (2023-2025) no se ha tocado. Ahora se mide una sola vez.

### Por qué walk-forward y no un ajuste único

Entrenar una vez con 2015-2022 y predecir tres años seguidos no es lo que haría un
operador real: él reentrena a medida que llegan datos. El **walk-forward** replica
eso: cada 30 días se reentrena con todo lo disponible hasta el día anterior al
bloque, y se predice el bloque siguiente. Nunca se usa un dato posterior al que se
predice.

Se comparan dos formas de ventana:

- **Expansiva:** el entrenamiento acumula toda la historia. Más datos.
- **Deslizante (3 años):** solo los últimos 3 años. Menos datos, pero se olvida de
  regímenes viejos. Con un mercado que cambió de régimen (El Niño 2023-2024),
  esto puede ayudar.

In [ ]:
REFIT_CADA = 30      # días entre reentrenamientos
MIN_ENTRENA = 730    # mínimo de días para entrenar


def walk_forward(modelo_fn, columnas, datos, indice_prueba, ventana="expansiva",
                 anios=3, etiqueta=""):
    pred = pd.Series(index=indice_prueba, dtype=float)
    fechas = list(indice_prueba)
    bloques = list(range(0, len(fechas), REFIT_CADA))

    for i in bloques:
        chunk = fechas[i:i + REFIT_CADA]
        corte = chunk[0] - pd.Timedelta(days=1)
        inicio = (corte - pd.DateOffset(years=anios) if ventana == "deslizante"
                  else datos.index.min())

        entrena = datos.loc[inicio:corte]
        if len(entrena) < MIN_ENTRENA:
            continue
        modelo = modelo_fn().fit(entrena[columnas], entrena["y"])
        pred.loc[chunk] = modelo.predict(datos.loc[chunk, columnas])

    print(f"  {etiqueta:28s} {len(bloques)} reentrenamientos")
    return pred


historia = datos.loc[:FIN_VALIDACION]   # todo lo anterior a la prueba
completo = datos                        # el walk-forward va creciendo sobre esto

predicciones = {}
print("Walk-forward sobre 2023-2025:")

predicciones["XGBoost expansiva"] = walk_forward(
    lambda: xgb.XGBRegressor(n_estimators=N_ARBOLES_SEL, **PARAMS),
    FEATURES_FINALES, completo, prueba.index, "expansiva", etiqueta="XGBoost expansiva")

predicciones["XGBoost deslizante 3a"] = walk_forward(
    lambda: xgb.XGBRegressor(n_estimators=N_ARBOLES_SEL, **PARAMS),
    FEATURES_FINALES, completo, prueba.index, "deslizante", etiqueta="XGBoost deslizante 3a")

predicciones["GAM (splines)"] = walk_forward(
    lambda: construir_gam(n_nodos),
    FEATURES_FINALES, completo, prueba.index, "expansiva", etiqueta="GAM expansiva")

# El alpha del Lasso se fijó en la ventana de selección y no se vuelve a buscar en
# cada reentrenamiento: buscarlo de nuevo con datos de la prueba sería fuga, y
# además multiplicaría por 37 el costo de la validación cruzada.
predicciones["Lasso"] = walk_forward(
    lambda: make_pipeline(StandardScaler(), Lasso(alpha=lasso.alpha_, max_iter=50000)),
    COLUMNAS, completo, prueba.index, "expansiva", etiqueta="Lasso expansiva")

predicciones["OLS votadas"] = walk_forward(
    lambda: make_pipeline(StandardScaler(), LinearRegression()),
    FEATURES_FINALES, completo, prueba.index, "expansiva", etiqueta="OLS votadas")

predicciones["Ingenuo (precio de ayer)"] = pd.Series(0.0, index=prueba.index)

### La prueba de Diebold-Mariano

Una diferencia de RMSE no basta: puede ser ruido muestral. La prueba de
Diebold-Mariano responde si la diferencia es **estadísticamente** significativa.

Con $d_t = e_{1t}^2 - e_{2t}^2$ la diferencia de pérdidas día a día:

$$DM = \frac{\bar d}{\sqrt{\widehat{\text{Var}}(\bar d)}} \;\xrightarrow{d}\; N(0,1)$$

- $DM < 0$ y $p < 0,05$: el modelo 1 es mejor, y la diferencia no es azar.
- $p > 0,05$: no hay evidencia de que uno sea mejor que el otro.

Se aplica la corrección de Harvey-Leybourne-Newbold para muestras finitas. **Esta
es la prueba que convierte "mi modelo tiene mejor RMSE" en un resultado
defendible.**

In [ ]:
def diebold_mariano(e1, e2, h=1):
    d = (e1 ** 2 - e2 ** 2).dropna()
    n = len(d)
    if n < 10:
        return np.nan, np.nan
    d_barra = d.mean()

    # Varianza de largo plazo (Newey-West con h-1 rezagos)
    gamma = [np.cov(d[k:], d[:n - k])[0, 1] if k else d.var(ddof=0) for k in range(h)]
    var_d = (gamma[0] + 2 * sum(gamma[1:])) / n
    if var_d <= 0:
        return np.nan, np.nan

    dm = d_barra / np.sqrt(var_d)
    correccion = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)   # Harvey-Leybourne-Newbold
    dm *= correccion
    p = 2 * (1 - stats.t.cdf(abs(dm), df=n - 1))
    return dm, p


errores = {m: (y_prueba - p).dropna() for m, p in predicciones.items()}
ingenuo = errores["Ingenuo (precio de ayer)"]

filas = []
for modelo, e in errores.items():
    if modelo == "Ingenuo (precio de ayer)":
        continue
    comun = e.index.intersection(ingenuo.index)
    dm, p = diebold_mariano(e.loc[comun], ingenuo.loc[comun])
    filas.append({"modelo": modelo, "DM": dm, "p": p,
                  "veredicto": ("gana el modelo" if p < 0.05 and dm < 0
                                else "gana el ingenuo" if p < 0.05
                                else "sin diferencia significativa")})
dm_tabla = pd.DataFrame(filas).set_index("modelo")
print("Diebold-Mariano contra el modelo ingenuo (2023-2025):")
dm_tabla.round(4)

In [ ]:
tabla_prueba = pd.DataFrame({m: evaluar_prueba(p.fillna(0))
                             for m, p in predicciones.items()}).T
print("MÉTRICAS FINALES EN PRUEBA (2023-2025) — se miden una sola vez")
tabla_prueba.round(4).sort_values("RMSE Δlog")

### Desempeño por régimen

Un promedio sobre tres años esconde que el modelo puede funcionar bien en calma y
mal en crisis. Se parte la prueba en terciles del precio observado y por año. El
tercil alto corresponde en buena parte a El Niño 2023-2024, que es el caso difícil
y el que realmente importa.

In [ ]:
mejor_modelo = tabla_prueba["RMSE Δlog"].idxmin()
pred_mejor = predicciones[mejor_modelo].fillna(0)

precio_prueba = precio.loc[prueba.index]
regimen = pd.qcut(precio_prueba, 3, labels=["bajo", "medio", "alto"])
grupos = {f"tercil {t}": regimen == t for t in ["bajo", "medio", "alto"]}
grupos.update({str(a): prueba.index.year == a for a in sorted(set(prueba.index.year))})

filas = []
for modelo in [mejor_modelo, "Ingenuo (precio de ayer)"]:
    p = predicciones[modelo].fillna(0)
    for nombre, mascara in grupos.items():
        idx = prueba.index[mascara]
        ev = construir_evaluador(idx, y_prueba.loc[idx])
        filas.append({"modelo": modelo, "grupo": nombre, "n": len(idx),
                      **ev(p.loc[idx])})
por_regimen = pd.DataFrame(filas).set_index(["modelo", "grupo"])
por_regimen.round(3)

In [ ]:
precio_ayer_prueba = precio.shift(1).loc[prueba.index]
nivel_pred = precio_ayer_prueba * np.exp(pred_mejor)
mape_movil = (100 * (nivel_pred - precio_prueba).abs() / precio_prueba).rolling(30).mean()

fig, ax = plt.subplots(3, 1, figsize=(13, 11))
ax[0].plot(precio_prueba, color="k", lw=0.8, label="real")
ax[0].plot(nivel_pred, color="tab:red", lw=0.8, label=mejor_modelo)
ax[0].set_title(f"Precio real vs pronóstico a un día — {mejor_modelo}"); ax[0].legend()

ax[1].scatter(y_prueba, pred_mejor, s=5, alpha=0.4)
lim = [y_prueba.min(), y_prueba.max()]
ax[1].plot(lim, lim, "k--", lw=0.8)
ax[1].axhline(0, color="gray", lw=0.5); ax[1].axvline(0, color="gray", lw=0.5)
ax[1].set_xlabel("Δlog real"); ax[1].set_ylabel("Δlog predicho")
ax[1].set_title("Cuadrantes 1 y 3 = dirección acertada")

ax[2].plot(mape_movil, color="tab:orange")
ax[2].set_title("MAPE móvil de 30 días (%)")
plt.tight_layout()
plt.show()

### ¿Qué variables usa el modelo?

- **Gain:** medida interna de XGBoost, cuánto mejora el ajuste cada vez que un
  árbol usa esa variable. Se calcula en entrenamiento, así que premia variables
  que el modelo usó mucho aunque no generalicen.
- **Permutación:** cuánto empeora el error **en prueba** al desordenar la variable.
  Es la medida confiable, porque se mide sobre datos que el modelo nunca vio.

Si una variable tiene mucho *gain* y poca importancia por permutación, el modelo la
estaba usando para memorizar, no para predecir.

In [ ]:
xgb_final = xgb.XGBRegressor(n_estimators=N_ARBOLES_SEL, **PARAMS).fit(
    historia[FEATURES_FINALES], historia["y"])

gain = (pd.Series(xgb_final.get_booster().get_score(importance_type="gain"))
        .reindex(FEATURES_FINALES).fillna(0))
perm = permutation_importance(xgb_final, prueba[FEATURES_FINALES], y_prueba,
                              n_repeats=10, random_state=SEMILLA,
                              scoring="neg_root_mean_squared_error")
perm = pd.Series(perm.importances_mean, index=FEATURES_FINALES)

fig, ax = plt.subplots(1, 2, figsize=(14, max(4, 0.35 * len(FEATURES_FINALES))))
perm.sort_values().plot.barh(ax=ax[0], color="tab:blue",
                             title="Importancia por permutación (en prueba)")
(gain / gain.sum()).sort_values().plot.barh(ax=ax[1], color="tab:green",
                                            title="Gain (normalizado, en entrenamiento)")
plt.tight_layout()
plt.show()

---
# 12. La dirección del precio como problema de clasificación

*(SLDA clases 12-14.)*

Hasta aquí el problema fue de regresión: ¿cuánto cambia el precio? Pero para
muchas decisiones basta con la **dirección**: ¿sube o baja? Reformularlo como
clasificación abre todo el instrumental de las clases 12-14, y además cambia la
función de pérdida: al modelo ya no le importa equivocarse por mucho en un día
raro, sino acertar el signo en muchos días.

$$z_t = \mathbb{1}\{y_t > 0\}$$

**Ojo con la línea base.** Si el 52% de los días el precio sube, un clasificador
que siempre diga "sube" acierta 52%. Esa, y no el 50%, es la cifra a superar.

In [ ]:
z_sel    = (y_sel > 0).astype(int)
z_val    = (y_val > 0).astype(int)
z_prueba = (y_prueba > 0).astype(int)

base_mayoritaria = max(z_sel.mean(), 1 - z_sel.mean())
print(f"Días que suben en selección: {100 * z_sel.mean():.1f}%")
print(f"Línea base (clase mayoritaria): {100 * base_mayoritaria:.1f}% de acierto")

### Regresión logística penalizada

Es `cv.glmnet(..., family="binomial")` del curso. La penalización $\ell_1$
selecciona variables igual que en regresión, pero ahora sobre la log-verosimilitud
binomial.

In [ ]:
# solver="saga" y no "liblinear": con 232 variables correlacionadas, liblinear tarda
# unos 16 minutos y saga llega al MISMO C y a las mismas variables en menos de 30 s.
logistica = LogisticRegressionCV(Cs=20, cv=tscv, penalty="l1", solver="saga",
                                 scoring="roc_auc", random_state=SEMILLA,
                                 max_iter=2000, n_jobs=-1)
logistica.fit(Xs_sel, z_sel)

coef_log = pd.Series(logistica.coef_[0], index=COLUMNAS)
sel_log = coef_log[coef_log != 0].abs().sort_values(ascending=False)
print(f"Logística L1: C = {logistica.C_[0]:.5f}, "
      f"{len(sel_log)} variables con coeficiente distinto de cero")
print("\nTop 10:")
print(sel_log.head(10).round(3).to_string())

arbol_clf = DecisionTreeClassifier(ccp_alpha=1e-3, random_state=SEMILLA).fit(
    sel[FEATURES_FINALES], z_sel)
xgb_clf = xgb.XGBClassifier(n_estimators=N_ARBOLES_SEL, **PARAMS).fit(
    sel[FEATURES_FINALES], z_sel)

### Curva ROC y AUC

La ROC recorre todos los umbrales de decisión y grafica la tasa de verdaderos
positivos contra la de falsos positivos. El **AUC** resume la curva en un número:
es la probabilidad de que el modelo le dé mayor puntaje a un día de subida que a
uno de bajada, elegidos al azar.

- AUC = 0,5: el modelo no distingue nada (la diagonal).
- AUC = 1,0: separación perfecta.

Ventaja sobre la tasa de acierto: **no depende del umbral** ni se deja engañar por
clases desbalanceadas.

In [ ]:
modelos_clf = {
    "Logística L1": (logistica.predict_proba(Xs_prueba)[:, 1], None),
    "Árbol podado": (arbol_clf.predict_proba(prueba[FEATURES_FINALES])[:, 1], None),
    "XGBoost": (xgb_clf.predict_proba(prueba[FEATURES_FINALES])[:, 1], None),
}

plt.figure(figsize=(6, 6))
filas = []
for nombre, (proba, _) in modelos_clf.items():
    fpr, tpr, _ = roc_curve(z_prueba, proba)
    auc = roc_auc_score(z_prueba, proba)
    plt.plot(fpr, tpr, lw=1.5, label=f"{nombre} (AUC = {auc:.3f})")
    filas.append({"modelo": nombre, "AUC": auc,
                  "Acierto (%)": 100 * accuracy_score(z_prueba, (proba > 0.5).astype(int))})

plt.plot([0, 1], [0, 1], "k--", lw=0.8, label="azar (AUC = 0.500)")
plt.xlabel("Tasa de falsos positivos"); plt.ylabel("Tasa de verdaderos positivos")
plt.title("Curva ROC — dirección del precio en prueba (2023-2025)")
plt.legend(loc="lower right")
plt.show()

clf_tabla = pd.DataFrame(filas).set_index("modelo")
clf_tabla.loc["Siempre 'sube'"] = [0.5, 100 * max(z_prueba.mean(), 1 - z_prueba.mean())]
clf_tabla.round(3)

### Matriz de confusión

El acierto global esconde **qué tipo** de error comete el modelo. Para un
comercializador no cuesta lo mismo perderse una subida que anticipar una que no
llegó.

In [ ]:
mejor_clf = clf_tabla.drop("Siempre 'sube'")["AUC"].idxmax()
proba_mejor = modelos_clf[mejor_clf][0]
pred_clase = (proba_mejor > 0.5).astype(int)

cm = confusion_matrix(z_prueba, pred_clase)
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]}\n({100 * cm[i, j] / cm.sum():.1f}%)",
                ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1], ["predice baja", "predice sube"])
ax.set_yticks([0, 1], ["realmente baja", "realmente sube"])
ax.set_title(f"Matriz de confusión — {mejor_clf}")
ax.grid(False)
plt.tight_layout()
plt.show()

vn, fp, fn, vp = cm.ravel()
print(f"Sensibilidad (detecta subidas):  {100 * vp / (vp + fn):.1f}%")
print(f"Especificidad (detecta bajadas): {100 * vn / (vn + fp):.1f}%")
print(f"Precisión cuando dice 'sube':    {100 * vp / (vp + fp):.1f}%")

---
# 13. Conclusiones

Esta celda resume los resultados que acaban de correr, no valores escritos a mano.

In [ ]:
mejor = tabla_prueba["RMSE Δlog"].idxmin()
ing = tabla_prueba.loc["Ingenuo (precio de ayer)"]
m = tabla_prueba.loc[mejor]
dm_mejor = dm_tabla.loc[mejor] if mejor in dm_tabla.index else None

print("=" * 70)
print("RESUMEN")
print("=" * 70)
print(f"Variables creadas:          {X.shape[1]}")
print(f"Tras limpieza:              {len(COLUMNAS)}")
print(f"Seleccionadas (>= {MIN_VOTOS} votos): {len(FEATURES_FINALES)}")
print()
print(f"Mejor modelo en prueba:     {mejor}")
print(f"  RMSE Δlog     {m['RMSE Δlog']:.4f}  vs ingenuo {ing['RMSE Δlog']:.4f}  "
      f"({100 * (m['RMSE Δlog'] / ing['RMSE Δlog'] - 1):+.1f}%)")
print(f"  MAPE nivel    {m['MAPE nivel (%)']:.2f}%  vs ingenuo {ing['MAPE nivel (%)']:.2f}%")
print(f"  Dirección     {m['Acierto dirección (%)']:.1f}%")
if dm_mejor is not None:
    print(f"  Diebold-Mariano contra el ingenuo: DM = {dm_mejor['DM']:.3f}, "
          f"p = {dm_mejor['p']:.4f}  ->  {dm_mejor['veredicto']}")
print()
print(f"Clasificación de dirección: {mejor_clf}, AUC = {clf_tabla.loc[mejor_clf, 'AUC']:.3f}")
print("=" * 70)

## Lo que hay que poder defender

1. **El objetivo es Δlog, no el nivel.** Predecir el nivel permite que un modelo
   que solo copia el precio de ayer se vea excelente. Predecir el cambio lo
   obliga a decir algo sobre el movimiento.
2. **Tres ventanas que no se traslapan.** Las variables se eligieron sin ver
   2020-2025; el modelo se eligió sin ver 2023-2025; la prueba se midió una vez.
3. **El bloque de despacho está excluido a propósito.** `costo_marginal_prog` *es*
   el precio de bolsa. Incluirlo daría un MAPE mucho mejor y una tesis sin
   contenido.
4. **Toda ventana móvil va con `shift(1)` antes del `rolling`.** Es la fuga más
   común y la más difícil de detectar a posteriori.
5. **Todo se compara contra el ingenuo, y con Diebold-Mariano.** Sin eso, una
   mejora de RMSE puede ser ruido muestral.
6. **Ningún selector decide solo.** Seis métodos votan; se conservan las variables
   con al menos tres votos.

## Extensiones naturales

- **Grouped lasso** (clase 9) con los bloques temáticos como grupos, para que el
  modelo incluya o excluya hidrología completa en vez de variables sueltas.
  Requiere una librería fuera de scikit-learn (`group-lasso` o `celer`).
- **Horizontes mayores a un día.** Todo aquí es a 1 día. A 7 o 30 días el ingenuo
  se vuelve mucho más débil y el aporte hidrológico debería crecer.
- **Intervalos de predicción** por regresión cuantílica, en vez de un solo número.
- **Tratamiento explícito de atípicos** (repo *feature-engineering*, sección 14):
  los picos por encima de 1.000 COP/kWh dominan el RMSE y podrían modelarse aparte.